# День 7: GRU побольше (hidden 192) на всех данных + усреднение эпох gru_full (~2.5 ч)
Ансамбль ft_swa + gru_full дал на лидерборде **0.65426** (+0.0034 к 0.65083 — ровно честная оценка смешивания). Из уже обученных моделей больше не выжать: остальные пять добавляли к паре ~0.0001.

Теперь numba-движок считает две модели за половину времени принятой GRU, поэтому модели можно сделать **больше**. Замер на моей машине (доля от времени GRU, прошедшей лимит):

| сабмит | время |
|---|---|
| ft_swa + gru_full (отправлен, 0.65426) | 0.53x |
| GRU-192 одна | 0.51x |
| ft_swa + GRU-192 | 0.79x |
| три модели по 128 | 0.80x |

Что делает ноутбук:
1. **Сразу (~15 мин):** проверяет, улучшает ли gru_full усреднение весов последних эпох (как ft_swa у бейзлайна) — бесплатный кандидат **A**.
2. **Обучение (~2 ч):** наша GRU с hidden 192 вместо 128 на всех 10 607 последовательностях, тот же рецепт, что у gru_full.
3. **Оценка и сборка (~20 мин):** все модели на одной валидации, кандидат **B** = ft_swa + лучшая из GRU-192; оба кандидата проверяются и скачиваются.

Перед запуском: **T4 GPU**; код на Drive (`wunder/day2`) уже свежий — тот, что вы загрузили для day6.

## 1. GPU + Drive + код + numba как у скорера

In [ ]:
!nvidia-smi --query-gpu=name --format=csv,noheader
from google.colab import drive
drive.mount('/content/drive')
W = '/content/drive/MyDrive/wunder'
!pip -q install "numba==0.67.0" onnx 2>&1 | grep -v "dependency resolver\|requires numba\|incompatible" || true
!rm -rf /content/day2 && cp -r $W/day2 /content/day2
%cd /content/day2
!python -c "import numba, numpy; print('numba', numba.__version__, '| numpy', numpy.__version__)"

## 2. Данные: train.parquet и valid.parquet из стартового пакета (~6 мин)

In [ ]:
import glob
URL = 'https://files.wundernn.io/wnn_connectome_starterpack.tar.gz'
!apt-get -qq update > /dev/null 2>&1; apt-get -qq install -y aria2 > /dev/null 2>&1
!aria2c -c -x 16 -s 16 -k 1M --file-allocation=none --summary-interval=30 --console-log-level=warn -d /content -o pack.tar.gz $URL
!mkdir -p /content/sp && tar -xzf /content/pack.tar.gz -C /content/sp --wildcards '*datasets/train.parquet' '*datasets/valid.parquet' 2>&1 | grep -v LIBARCHIVE
!rm -f /content/pack.tar.gz
TRAIN_PQ = glob.glob('/content/sp/**/datasets/train.parquet', recursive=True)[0]
VALID_PQ = glob.glob('/content/sp/**/datasets/valid.parquet', recursive=True)[0]
import pyarrow.parquet as pq
print('train sequences:', pq.ParquetFile(TRAIN_PQ).num_row_groups, '| valid sequences:', pq.ParquetFile(VALID_PQ).num_row_groups)

## 3. Валидация — те же 600 последовательностей `valid_sub` (~1.5 мин)

In [ ]:
!python prepare_data.py --src $W/colab_data/valid_sub.parquet --out /content/data/valid

## 4. Кандидат A: усреднение эпох gru_full (~15 мин)
У дообученного бейзлайна усреднение весов эпох 2–5 (ft_swa) оказалось лучшей одиночной моделью. Проверяем то же для gru_full: эпохи 4–7 и 5–7 против `best.pt`. Смотрите на блок `best equal-weight blends`: строки `ft_swa+gru_full…`.

In [ ]:
import os
def swa(run, ks):
    return '+'.join(f'{run}/epoch_{k}.pt' for k in ks)
GF, G192 = W + '/runs/gru_full', W + '/runs/gru192_full'
CANDS = {
    'ft_swa':          swa(W + '/runs/ft_baseline_full', range(2, 6)),
    'gru_full':        GF + '/best.pt',
    'gru_full_swa4_7': swa(GF, range(4, 8)),
    'gru_full_swa5_7': swa(GF, range(5, 8)),
}
def models_arg(cands):
    found = {k: v for k, v in cands.items() if all(os.path.exists(p) for p in v.split('+'))}
    print('models:', list(found), '| missing:', [k for k in cands if k not in found])
    return found, ' '.join(f'--model {k}={v}' for k, v in found.items())
FOUND, MODELS = models_arg(CANDS)
!python ensemble_eval.py --valid-dir /content/data/valid --out /content/preds7 $MODELS --report $W/day7_swa.json

## 5. Обучение GRU-192 на всех данных (~2 ч)
Тот же рецепт, что у gru_full (8 эпох, lr 2e-3, батч 64, куски по 1000 шагов), только `--hidden 192`: параметров вдвое больше (430 тыс. против 220 тыс.). Каждая эпоха сохраняется на Drive.

Если сессия оборвалась: ячейки 1–3, затем эту же ячейку с `--resume` в конце команды — обучение продолжится с последней сохранённой эпохи.

In [ ]:
!python train.py --train-parquet $TRAIN_PQ --valid-dir /content/data/valid --out $G192 \
    --arch gru --hidden 192 --epochs 8 --batch 64 --chunk 1000 --lr 2e-3 --keep-epochs

## 6. Все модели на одной валидации (~10 мин; прогнозы моделей из шага 4 берутся из кэша)

In [ ]:
CANDS.update({'gru192': G192 + '/best.pt', 'gru192_swa5_7': swa(G192, range(5, 8))})
FOUND, MODELS = models_arg(CANDS)
!python ensemble_eval.py --valid-dir /content/data/valid --out /content/preds7 $MODELS --report $W/day7_all.json

## 7. Сборка кандидатов (~15 мин)
- **A** = ft_swa + лучший вариант gru_full (по смеси с равными весами). Собирается, только если усреднение эпох лучше `best.pt` — иначе A совпадает с уже отправленным сабмитом. Время как у отправленного (0.53x).
- **B** = ft_swa + лучший вариант GRU-192. Собирается, только если он лучше A. Время ~0.8x.

In [ ]:
import json
eb = json.load(open(W + '/day7_all.json'))['equal_blends']
def best(partners):
    keys = [f'ft_swa+{p}' for p in partners if f'ft_swa+{p}' in eb]
    return max(keys, key=eb.get) if keys else None
A = best(['gru_full', 'gru_full_swa4_7', 'gru_full_swa5_7'])
B = best(['gru192', 'gru192_swa5_7'])
print('sent already: ft_swa+gru_full', eb.get('ft_swa+gru_full'))
print('A:', A, eb.get(A), '| B:', B, eb.get(B))
BUILD_A = A is not None and A != 'ft_swa+gru_full'
BUILD_B = B is not None and eb[B] > max(eb.get(A, 0), eb.get('ft_swa+gru_full', 0))
def spec(key):
    return ' '.join(f'--model "{n}={FOUND[n]}@0.5"' for n in key.split('+'))
A_OUT = W + '/submissions/ens7_A_' + (A or '').split('+')[-1]
B_OUT = W + '/submissions/ens7_B_' + (B or '').split('+')[-1]
A_SPEC, B_SPEC = (spec(A) if A else ''), (spec(B) if B else '')
REF = W + '/submissions/gru128x2_v1'
print('build A:', BUILD_A, '| build B:', BUILD_B)

In [ ]:
if BUILD_A:
    !python build_ensemble.py --valid-dir /content/data/valid --out $A_OUT $A_SPEC
    !python check_submission.py {A_OUT}.zip --data $W/colab_data/valid_sub.parquet --seqs 3 --rows 20000 --ref-dir $REF
if BUILD_B:
    !python build_ensemble.py --valid-dir /content/data/valid --out $B_OUT $B_SPEC
    !python check_submission.py {B_OUT}.zip --data $W/colab_data/valid_sub.parquet --seqs 3 --rows 20000 --ref-dir $REF

## 8. Скорость кандидатов против принятой GRU (~2 мин) и скачивание

In [ ]:
OUTS = [o for o, b in ((A_OUT, BUILD_A), (B_OUT, BUILD_B)) if b]
if OUTS:
    OUTS_STR = ' '.join(OUTS)
    !python bench_speed.py $REF $OUTS_STR --rows 5000 --repeats 5
    from google.colab import files
    for o in OUTS:
        files.download(o + '.zip')
else:
    print('nothing better than the submitted ensemble - send me the output of cells 4 and 6')

## Что прислать
Вывод ячеек 4, 6 и 7–8 целиком. Если собрались оба кандидата, отправляйте сначала **A** (по времени он такой же, как уже прошедший сабмит), затем **B**. И если платформа показывает время работы сабмита — пришлите его для 0.65426: по нему видно, сколько запаса для B и для трёх моделей.